# Wine Price Prediction with Native CatBoost Text Features

Wine Reviewsデータの `description` を外部encoderでembeddingせず、CatBoostの `text_features` としてそのまま入力し、数値・カテゴリ変数とともに $\log(\mathrm{price})$ を予測します。

CatBoostはテキストを内部でtokenizeし、辞書とBoW・NaiveBayes・BM25などのtext calcerを利用して数値特徴へ変換します。これは事前学習LLMの知識を利用する方法ではなく、訓練データからテキスト統計を学習する方法です。

### 実行コードの説明

CatBoost、データ処理、可視化、データ分割、評価に必要なライブラリを読み込みます。`text_features` は `fit` の引数で列名を指定するため、専用の外部embeddingライブラリは使用しません。

In [1]:
# CatBoostが未インストールの場合のみコメントを外してください。
!pip install -q -U catboost

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")
SEED = 42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


## 1. データの読み込みと重複除外

### 実行コードの説明

`wine_review_CatBoost.ipynb` と比較できるよう、同じ `title` と `description` の組で重複を除外します。本文と価格があり、価格が正の行だけをモデル対象とします。

In [2]:
filename = "winemag-data-130k-v2.csv"
candidate_paths = [Path("data") / filename, Path("../data") / filename]
csv_path = next((path for path in candidate_paths if path.exists()), None)

if csv_path is None:
    searched = "\n".join(f"- {path.resolve()}" for path in candidate_paths)
    raise FileNotFoundError(f"CSV が見つかりません。以下を確認してください。\n{searched}")

df = pd.read_csv(csv_path)
rows_before = len(df)
df = df.drop_duplicates(subset=["title", "description"]).copy()
duplicates_removed = rows_before - len(df)

print(f"Loaded: {csv_path.resolve()}")
print(f"Removed duplicated reviews: {duplicates_removed:,}")
print(f"Shape after deduplication: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

Loaded: /content/data/winemag-data-130k-v2.csv
Removed duplicated reviews: 9,983
Shape after deduplication: 119,988 rows x 14 columns


## 2. 数値・カテゴリ・テキスト特徴量と目的変数

### 実行コードの説明

評価点を数値特徴量、産地・品種・ワイナリーなどをカテゴリ特徴量、`description` をテキスト特徴量として指定します。カテゴリ欠損は専用カテゴリへ置換し、本文は文字列型へ統一します。目的変数には価格の自然対数を使います。

In [ ]:
numeric_features = ["points"]
categorical_features = [
    "country",
    "designation",
    "province",
    "region_1",
    "region_2",
    "variety",
    "winery",
]
text_features = ["description"]
feature_columns = numeric_features + categorical_features + text_features

model_data = df.loc[
    df["description"].notna()
    & df["price"].notna()
    & (df["price"] > 0),
    feature_columns + ["price"],
].copy()

for column in categorical_features:
    model_data[column] = model_data[column].fillna("__MISSING__").astype(str)
for column in text_features:
    model_data[column] = model_data[column].astype(str)

X = model_data[feature_columns]
y = np.log(model_data["price"]).astype("float32")

print(f"Rows used for modeling: {len(model_data):,}")
print(f"Feature columns: {feature_columns}")
display(X.head())
display(y.describe().rename("log_price").to_frame())

## 3. Train / validation / test分割と学習

### 実行コードの説明

80%を訓練、10%を検証、10%を最終テストへ固定乱数で分割します。`cat_features` と `text_features` に列名を渡すことで、CatBoostがカテゴリ処理とテキスト数値化を内部で行います。検証RMSEが100 iteration改善しなければ停止し、最良モデルを復元します。

In [4]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.5, random_state=SEED
)

print(f"Train: {len(X_train):,}, validation: {len(X_valid):,}, test: {len(X_test):,}")

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=2_000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    text_features=text_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
)

print(f"Best iteration: {model.get_best_iteration():,}")

Train: 89,274, validation: 11,159, test: 11,160
0:	learn: 0.6407617	test: 0.6400051	best: 0.6400051 (0)	total: 4.66s	remaining: 2h 35m 12s
100:	learn: 0.3577797	test: 0.3423570	best: 0.3423570 (100)	total: 2m 12s	remaining: 41m 24s
200:	learn: 0.3465508	test: 0.3320305	best: 0.3320305 (200)	total: 4m 6s	remaining: 36m 48s
300:	learn: 0.3379045	test: 0.3247588	best: 0.3247588 (300)	total: 6m 3s	remaining: 34m 14s
400:	learn: 0.3321633	test: 0.3209119	best: 0.3209119 (400)	total: 7m 58s	remaining: 31m 46s
500:	learn: 0.3276618	test: 0.3183425	best: 0.3183425 (500)	total: 9m 52s	remaining: 29m 32s
600:	learn: 0.3240445	test: 0.3166977	best: 0.3166977 (600)	total: 11m 48s	remaining: 27m 29s


KeyboardInterrupt: 

## 4. テストデータでの評価

### 実行コードの説明

モデル選択に使っていないテストデータからlog価格を予測し、RMSE、MAE、$R^2$ を計算します。さらに指数変換してドル価格へ戻したMAEも表示します。

In [5]:
y_pred = model.predict(X_test)
metrics = pd.Series({
    "RMSE (log price)": mean_squared_error(y_test, y_pred) ** 0.5,
    "MAE (log price)": mean_absolute_error(y_test, y_pred),
    "R2 (log price)": r2_score(y_test, y_pred),
    "MAE (price, USD)": mean_absolute_error(np.exp(y_test), np.exp(y_pred)),
})
display(metrics.to_frame("value"))

CatBoostError: There is no trained model to use predict(). Use fit() to train model. Then use this method.

### 予測値と残差を可視化するコード

左図で実際のlog価格と予測log価格を比較し、右図で「実際値 − 予測値」の分布を確認します。45度線からの距離と残差分布から、誤差の大きさや系統的な偏りを判断します。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(x=y_test, y=y_pred, alpha=0.25, s=20, ax=axes[0])
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(limits, limits, "--", color="black")
axes[0].set(
    title="Actual vs. Predicted Log Price",
    xlabel="Actual log(price)",
    ylabel="Predicted log(price)",
)

residuals = y_test - y_pred
sns.histplot(residuals, bins=50, ax=axes[1])
axes[1].axvline(0, linestyle="--", color="black")
axes[1].set(title="Residual Distribution", xlabel="Actual - predicted")

plt.tight_layout()
plt.show()

## 5. 特徴量重要度

### 実行コードの説明

元の数値・カテゴリ・テキスト列単位でCatBoostの特徴量重要度を取得します。`description` の重要度は、内部で作成された複数のテキスト特徴を元の入力列へ集約した値として解釈します。重要度は因果関係を示すものではありません。

In [ ]:
feature_importance = (
    pd.Series(
        model.get_feature_importance(),
        index=feature_columns,
        name="importance",
    )
    .sort_values(ascending=False)
)
display(feature_importance.to_frame())

plt.figure(figsize=(9, 6))
feature_importance.sort_values().plot.barh()
plt.title("CatBoost Feature Importances with Native Text Feature")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 6. 予測結果の例

### 実行コードの説明

テストデータの先頭10件について、レビュー本文、表形式特徴、実価格、予測価格を表示します。テキストが長いため、表示幅を制限しながら個別の過大・過小予測を確認します。

In [ ]:
prediction_examples = X_test.head(10).copy()
prediction_examples["actual_price"] = np.exp(y_test.head(10))
prediction_examples["predicted_price"] = np.exp(model.predict(X_test.head(10)))
display(prediction_examples)

## 注意点

- CatBoostの `text_features` はテキストを直接受け取れますが、事前学習済みLLMの一般知識や文脈embeddingを利用するものではありません。
- 外部encoder方式、Ridge stacking方式、ネイティブtext方式を比較する際は、同じ重複除外、データ分割、CPU/GPU条件を使用してください。
- CatBoostの既定text processingはバージョンにより変わる可能性があります。再現性を厳密に管理する場合は、`tokenizers`、`dictionaries`、`feature_calcers` または `text_processing` を明示してください。